## ⏳ Time Travel en Delta Lake

#### ¿Sabías que una Delta Table puede volver a una versión específica de su pasado?

Una de las capacidades más interesantes de Delta Lake es **Time Travel**, una funcionalidad que permite consultar el estado histórico de una tabla sin necesidad de realizar respaldos manuales.

Esta característica es posible gracias al **Transaction Log (_delta_log)**, el componente encargado de registrar cada operación realizada sobre una Delta Table. Gracias a este historial transaccional podemos acceder a versiones anteriores de los datos mediante dos enfoques:

* #### 🔢 Por versión: 
    - Consultar una versión específica registrada en el historial de la tabla.

* #### 🕒 Por fecha y hora
    - Consultar el estado que tenía la tabla en un momento exacto del tiempo.
---

#### 🚀 ¿Por qué es importante?

Time Travel resulta especialmente útil para:

* ✅ Auditorías de datos
* ✅ Análisis de cambios históricos
* ✅ Comparación entre versiones
* ✅ Validación de procesos ETL
* ✅ Recuperación de información ante modificaciones no deseadas

En este notebook exploraremos ambos enfoques de forma práctica para comprender cómo Delta Lake conserva y gestiona el historial de una tabla.

### 🚀 Punto de Inicio en Databricks

Antes de trabajar con Delta Lake necesitamos una sesión de Spark activa.

Spark será el motor encargado de:

* ✅ Leer datos
* ✅ Transformarlos
* ✅ Procesarlos de forma distribuida
* ✅ Persistirlos como Delta Tables

In [0]:
from pyspark.sql import SparkSession # Puerta de entrada para trabajar con spark <-- SIEMPRE DEBEMOS IMPORTAR LA LLAVE MAESTRA QUE INICIA TODO.
from pyspark.sql.functions import *  # Funciones propias del módulo SQL de Spark, para trabajar sobre Dataframes.
spark = SparkSession.builder.appName("02TimeTravel").getOrCreate() 
"""
^          ^__________^        ^_________^                               ^
|                |                   |                                   | 
Variable   Constructor de Sesión   Nombre Aplicación       Evita conflicto del SparkSession"""

print("🚀 Spark Session iniciada correctamente")

#### 🔍 ¿Qué acaba de ocurrir?

Acabamos de crear una Spark Session. Esta sesión representa nuestro punto de entrada hacia:
* 🏗️ Apache Spark
* 🏗️ Databricks Runtime
* 🏗️ Delta Lake
* 🏗️ Unity Catalog

### ⚡ Utilizaremos la Delta Table del notebook anterior

- Nombre: `customers_delta_table`

In [0]:
## LECTURA INICIAL DE SU INFORMACIÓN
display(spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table"))
### ➡️ Resultado: 1 Solo registro

##================================================

## REVISAMOS SU HISTORIAL
display(spark.sql("DESCRIBE HISTORY catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table"))
### ➡️ Resultado: 2 Versiones

##================================================

## REGISTRAMOS DATOS (SE MODIFICA LA CANTIDAD DE REGISTROS Y SE REGISTRA UNA NUEVA VERSIÓN AL TRANSACTION LOG)

spark.sql("""
          INSERT INTO catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table
          VALUES 
          (2, 'Rafael','Bolaños','USA','rafael@gmail.com'),
          (3,'Pepito','Perez','Brazil','pepito@gmail.com');
          """)
print("Datos registrados correctamente")

#### 🔢 Time Travel por Versión

* Cada operación realizada sobre una Delta Table genera una nueva versión dentro de su historial transaccional.

* Gracias a ello, podemos consultar cómo lucían los datos en una versión específica de la tabla, independientemente de los cambios posteriores.

* Durante esta demostración accederemos a distintas versiones registradas por Delta Lake para observar cómo evoluciona la información a lo largo del tiempo.

📜 Verificamos el historial mediante: `DESCRIBE HISTORY`

🎯 Objetivo: recuperar el estado exacto de una versión determinada.

In [0]:
## REVISAMOS SU HISTORIAL
# display(spark.sql("DESCRIBE HISTORY catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table"))
### ➡️ Resultado: 3 Versiones

##================================================

## LECTURA DE SU INFORMACIÓN (VERSIÓN 0)
display(spark.sql("""
                  SELECT * 
                  FROM 
                  catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table
                  VERSION AS OF 0
                  """))
### ➡️ Resultado: Sin registros - Debido que la versión 0 es la creación de la tabla.



#### 🕒 Time Travel por Fecha y Hora

* Además de consultar versiones numéricas, Delta Lake permite recuperar el estado histórico de una tabla utilizando una fecha y hora específica.

* Internamente, Databricks identifica cuál era la versión activa en ese instante y reconstruye los datos correspondientes.

* Este enfoque resulta especialmente útil cuando conocemos el momento en que ocurrió un evento, pero no necesariamente la versión asociada.

📜 Verificamos el historial mediante: `DESCRIBE HISTORY`

🎯 Objetivo: consultar la información tal como existía en un instante concreto del historial de la tabla.


In [0]:
## REVISAMOS SU HISTORIAL
# display(spark.sql("DESCRIBE HISTORY catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table"))
### ➡️ Resultado: 3 Versiones

##================================================

## LECTURA DE SU INFORMACIÓN (FECHA Y HORA EN FORMATO TIMESTAMP)
display(spark.sql("""
                  SELECT * 
                  FROM 
                  catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table
                  TIMESTAMP AS OF '2026-06-11T04:41:15.000+00:00'
                  """))
### ➡️ Resultado: 1 registro - Debido que el registro pertenece a esa fecha y hora.
